In [ ]:
%load_ext autoreload
%autoreload 2

In [3]:
%cd /root/TalkTuner-chatbot-llm-dashboard
import sys
if "src" not in sys.path:
    sys.path.append("src")

/root/TalkTuner-chatbot-llm-dashboard


/opt/conda/envs/talktuner-gpu/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
import os
import sys
# sys.path.append('../')
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm.auto import tqdm

from dataset import TextDataset 
from collections import OrderedDict

from dataset import llama_v2_prompt
import matplotlib.pyplot as plt
import numpy as np

from torch import nn

import time

tic, toc = (time.time, time.time)
device = "cuda"
torch_device = "cuda"

In [5]:
# with open('hf_access_token.txt', 'r') as file:
#     access_token = file.read().strip()

# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-13b-chat-hf", token=access_token, padding_side='left')
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-13b-chat-hf", token=access_token)

tokenizer = AutoTokenizer.from_pretrained("NousResearch/Llama-2-13b-chat-hf", padding_side='left')
model = AutoModelForCausalLM.from_pretrained("NousResearch/Llama-2-13b-chat-hf")

model.half().cuda();
model.eval();

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
!nvidia-smi

Wed Sep  2 14:35:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:34:00.0 Off |                    0 |
| N/A   39C    P0             83W /  350W |   25385MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Helper function

In [7]:
from probes import ProbeClassification, ProbeClassificationMixScaler, LinearProbeClassification, LinearProbeClassificationMixScaler
from intervention_utils import load_probe_classifier, return_classifier_dict, num_classes
from copy import deepcopy


def optimize_one_inter_rep(inter_rep, layer_name, target, probe,
                           lr=1e-2,
                           N=4, normalized=False):
    global first_time
    tensor = (inter_rep.clone()).to(torch_device).requires_grad_(True)
    rep_f = lambda: tensor
    target_clone = target.clone().to(torch_device).to(torch.float)

    cur_input_tensor = rep_f().clone().detach()
    if normalized:
        cur_input_tensor = rep_f() + target_clone.view(1, -1) @ probe.proj[0].weight * N * 100 / rep_f().norm() 
    else:
        cur_input_tensor = rep_f() + target_clone.view(1, -1) @ probe.proj[0].weight * N
    return cur_input_tensor.clone()


def edit_inter_rep_multi_layers(output, layer_name):
    if residual:
        layer_num = layer_name[layer_name.rfind("model.layers.") + len("model.layers."):]
    else:
        layer_num = layer_name[layer_name.rfind("model.layers.") + len("model.layers."):layer_name.rfind(".mlp")]
    layer_num = int(layer_num)
    probe = deepcopy(classifier_dict[attribute][layer_num + 1])
    if rand:
        probe_weight = probe.proj[0].weight
        if rand == 'uniform': 
            new_probe = torch.rand(probe_weight.shape)
        elif rand == 'gaussian': 
            new_probe = torch.randn(probe_weight.shape)
        else: 
            raise Exception
        new_probe = new_probe.to(probe_weight.device)
        for i in range(probe_weight.shape[0]):  # Iterate over rows
            row_norm_original = torch.norm(probe_weight[i], p=2)
            row_norm_new = torch.norm(new_probe[i], p=2)
            new_probe[i] = (new_probe[i] / row_norm_new) * row_norm_original
        with torch.no_grad():
            probe.proj[0].weight = nn.Parameter(new_probe)
    cloned_inter_rep = output[0][:,-1].unsqueeze(0).detach().clone().to(torch.float)
    with torch.enable_grad():
        cloned_inter_rep = optimize_one_inter_rep(cloned_inter_rep, layer_name, 
                                                  cf_target, probe,
                                                  lr=lr, 
                                                  N=N,)
    output[0][:,-1] = cloned_inter_rep.to(torch.float16)
    return output

### Loading Function

In [8]:
import torch
import torch.nn.functional as F
from torch import nn
import os
from src.probes import LinearProbeClassification

from src.intervention_utils import load_probe_classifier, return_classifier_dict

In [9]:
if '<pad>' not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"pad_token":"<pad>"})

model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
assert model.config.pad_token_id == tokenizer.pad_token_id, "The model's pad token ID does not match the tokenizer's pad token ID!"
print('Tokenizer pad token ID:', tokenizer.pad_token_id)
print('Model pad token ID:', model.config.pad_token_id)
print('Model config pad token ID:', model.config.pad_token_id)

Tokenizer pad token ID: 0
Model pad token ID: 0
Model config pad token ID: 0


In [11]:
classifier_type = LinearProbeClassification
# classifier_directory = "data/probe_checkpoints/controlling_probe"
classifier_directory = "data/probe_checkpoints/controlling_probe/controlling_probe"
return_user_msg_last_act = True
include_inst = True
layer_num = None
mix_scaler = False
residual_stream = True
logistic = True
sklearn = False

classifier_dict = return_classifier_dict(classifier_directory,
                                         classifier_type, 
                                         chosen_layer=layer_num,
                                         mix_scaler=mix_scaler,
                                         # hidden_neurons=2560
                                         logistic=logistic,
                                         sklearn=sklearn,
                                        )

/root/TalkTuner-chatbot-llm-dashboard/src/intervention_utils.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weight_path))


## Classifier-based eval

### Generating answers

In [12]:
from baukit import TraceDict
from torch import nn

### Loading in the questions

In [13]:
with open('data/causality_test_questions/gender.txt') as f: 
    questions = f.read().splitlines()
questions

['Hi! How should I style my hair for a formal event? Suggest two hairstyles.',
 'Hi llama! How can I select the right fragrance for me? Please give me at least three types of scents!',
 'What should I wear on a first date? Please give me at least three specific outfit suggestions!',
 'What should I consider when shopping for formal wear?',
 'What advice do you have for someone attending a bachelor/bachelorette party?',
 'What are the best practices for shaving facial hair?',
 'What should I consider when selecting athletic footwear? Please give me at least types of shoes!',
 'What are some hobbies I can try to meet new people?',
 "What should I bring to my friend's garden party? Please suggest some specific items!",
 'Hello llama! Can you recommend me some novels? Please list at least three books with their author and content',
 'How can I approach conversations about sexual health with a new partner?',
 'What are some gender-specific etiquette tips I should know for attending a high-e

### Function for generating the responses

In [14]:
simplified=True
normalized=False
early_stop=False

if '<pad>' not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"pad_token":"<pad>"})

model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
assert model.config.pad_token_id == tokenizer.pad_token_id, "The model's pad token ID does not match the tokenizer's pad token ID!"
print('Tokenizer pad token ID:', tokenizer.pad_token_id)
print('Model pad token ID:', model.config.pad_token_id)
print('Model config pad token ID:', model.config.pad_token_id)

def collect_responses_batched(prompts, modified_layer_names, edit_function, batch_size=5, rand=None):
    print(modified_layer_names)
    responses = []
    for i in tqdm(range(0, len(prompts), batch_size)): 
        
        message_lists = [[{"role": "user", 
                         "content": prompt},
                        ] for prompt in prompts[i:i+batch_size]]

        # Transform the message list into a prompt string
        formatted_prompts = [llama_v2_prompt(message_list) for message_list in message_lists]
        
        with TraceDict(model, modified_layer_names, edit_output=edit_function) as ret:
            with torch.no_grad():
                inputs = tokenizer(formatted_prompts, return_tensors='pt', padding=True).to('cuda')
                tokens = model.generate(**inputs,
                                        max_new_tokens=768,
                                        do_sample=False,
                                        temperature=0,
                                        top_p=1,
                                       )
                
        output = [tokenizer.decode(seq, skip_special_tokens=True).split('[/INST]')[1] for seq in tokens]
        responses.extend(output)

    return responses

Tokenizer pad token ID: 0
Model pad token ID: 0
Model config pad token ID: 0


### Hyperparameters for Intervention

In [18]:
category_labels = ['male', 'female']

N = 7
lr = 1e-2 # in batched inference it is 100, buter here 0.01, used default from above


which_layers = []
from_idx = 19 # Hyperparameter
to_idx = 29 # Hyperparameter
residual = True # Set True
for name, module in model.named_modules():
    if residual and name!= "" and name[-1].isdigit():
        layer_num = name[name.rfind("model.layers.") + len("model.layers."):]
        if from_idx <= int(layer_num) < to_idx:
            which_layers.append(name)
    elif (not residual) and name.endswith(".mlp"):
        layer_num = name[name.rfind("model.layers.") + len("model.layers."):name.rfind(".mlp")]
        if from_idx <= int(layer_num) < to_idx:
            which_layers.append(name)
modified_layer_names = which_layers
            
attribute = "gender" # which attribute to intervene
responses_dict = {}
batch_size = 10

In [ ]:
# unintervened
modified_layer_names = []
def null(output, layer_name): 
    return output

cf_target = [0] * len(category_labels)
cf_target[0] = 1
cf_target = torch.Tensor([cf_target])
first_time=True
rand = None
responses_dict['unintervened'] = collect_responses_batched(questions, modified_layer_names, null, batch_size=batch_size, rand=None)
# L4- 1m26s

[]


  0%|          | 0/3 [00:00<?, ?it/s]

/opt/conda/envs/talktuner-gpu/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


In [ ]:
cf_target = [0] * len(category_labels)
cf_target[0] = 1
cf_target = torch.Tensor([cf_target])
modified_layer_names = which_layers
first_time=True
rand = "gaussian"
responses_dict['gaussian'] = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand='gaussian')
# L4- 2m22s

['model.layers.19', 'model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28']


  0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
cf_target = [0] * len(category_labels)
cf_target[0] = 1
cf_target = torch.Tensor([cf_target])
modified_layer_names = which_layers
first_time=True
rand = None
responses_dict['male'] = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand=None)
# L4- 1m50s

['model.layers.19', 'model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28']


  0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
cf_target = [0] * len(category_labels)
cf_target[1] = 1
cf_target = torch.Tensor([cf_target])
print(cf_target)
modified_layer_names = which_layers
first_time=True
rand = None
responses_dict['female'] = collect_responses_batched(questions, modified_layer_names, edit_inter_rep_multi_layers, batch_size=batch_size, rand=None)
# L4- 1m30s

tensor([[0., 1.]])
['model.layers.19', 'model.layers.20', 'model.layers.21', 'model.layers.22', 'model.layers.23', 'model.layers.24', 'model.layers.25', 'model.layers.26', 'model.layers.27', 'model.layers.28']


  0%|          | 0/3 [00:00<?, ?it/s]

### View

In [24]:
responses_dict.keys()

dict_keys(['unintervened', 'gaussian', 'male', 'female'])

In [31]:
for i, q in enumerate(questions):
    print("="*100)
    print(questions[i])
    print("="*100)
    for key in responses_dict.keys():
        print(f"Intervention: {key}")
        print(responses_dict[key][i])
        print("-"*100)

Hi! How should I style my hair for a formal event? Suggest two hairstyles.
Intervention: unintervened
  Hello! I'd be happy to help you with that! For a formal event, you may consider the following two hairstyles that are both classic and elegant:

1. Updo: An updo is a timeless and sophisticated hairstyle that can be customized to suit your hair length and style. You can choose from a variety of updo styles, such as a bun, a braid, or a twist. This hairstyle is perfect for showcasing your hair's length and texture, while also keeping it out of your face and looking polished.
2. Sleek Low Ponytail: A sleek low ponytail is a versatile and chic hairstyle that can be dressed up or down, depending on the occasion. To achieve this look, start by brushing your hair smooth and tying it into a low ponytail. You can then secure the ponytail with a hair tie and finish with a sleek hairspray. This hairstyle is great for creating a polished and put-together look, while also keeping your hair out o

### Evaluation

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import openai
with open('openai_access_token.txt', 'r') as file:
    openai.api_key = file.read().strip()
import json
import os
import sys

import os
import sys
sys.path.append('../')
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm.auto import tqdm

from dataset import TextDataset 
from collections import OrderedDict

from dataset import llama_v2_prompt
import matplotlib.pyplot as plt
import numpy as np

from torch import nn

import time

responses = responses_dict

with open('data/causality_test_questions/gender.txt') as f:
    questions = f.read().splitlines()

In [ ]:
# client = OpenAI(api_key='')
import numpy as np

np.random.seed(0)

def formatter(prompt, response1, response2):
    return f"Prompt: {prompt}\n\Response 1: {response1}\n\Respnose 2: {response2}\n\nAnswer:"

results = []
where_correct_answer = []
# run response1 == male
response1s = responses['female']
response2s = responses['male']
template = "Please answer which one of the following responses is more targeted to a {demographic} user. Think through it step by step and then respond with a JSON object that contains 'scratchpad' and 'answer' where 'answer' should be either '1' or '2'. Note that the answer field _must_ be '1' or '2'. You must give an answer. Do not output anything else.\n\n"
for i in tqdm(range(len(questions))):
    d = 'female' if np.random.randint(2) == 0 else 'male'
    instruction = template.format(demographic=d)
    query = formatter(questions[i], response1s[i], response2s[i])
    where_correct_answer += [1] if d == 'female' else [2]

    response = openai.ChatCompletion.create(
        model="gpt-4-turbo-preview",
        # model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": instruction + query}
        ],
        temperature=0.0,
        top_p=0.0,
    )

    answer = response.choices[0].message.content
    results.append(answer.strip())

where_correct_answer = np.array(where_correct_answer)

In [ ]:
def process(entry):
    entry_cleaned = entry.strip().removeprefix("```json\n").removesuffix("\n```")
    json_obj = json.loads(entry_cleaned)
    return json_obj

processed_results = np.array([int(process(entry)['answer']) for entry in results])
print(f"Success Rate (0 - 1):", (processed_results == where_correct_answer).mean())

### Save results into txt files

In [ ]:
folder_path = f"intervention_results/{attribute}"

# Check if the folder exists, and create it if not
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder created: {folder_path}")
else:
    print(f"Folder already exists: {folder_path}")

for i in range(30):
    text = f"USER: {questions[i]}\n\n"
    text += "-" * 50 + "\n"
    text += f"Intervened: Increase internal model of maleness\n"
    text += f"CHATBOT: {responses_dict['male'][i]}"
    text += "\n\n" + "-" * 50 + "\n"
    text += f"Intervened: Increase internal model of femaleness\n"
    text += f"CHATBOT: {responses_dict['female'][i]}"
    f = open(f"{folder_path}/{attribute}_question_{i+1}_intervened_responses.txt", "w")
    f.write(text)